# **04 - Build `artists` table (Last.fm + MusicBrainz) for Gametime shows**
***

**What:** Adds information about each *artist* (how popular, what genre, how long they have been around) so the model can use it to predict ticket prices.

**Why:** Gametime tells us what tickets cost for each show, but not who the artist is. Price depends a lot on the artist: a huge act sells for far more than a local band. This notebook finds that information and saves it in one table.

**What it does:**
1. Reads the Gametime shows from the database and cleans each show name into an artist name (for example, "Kelly Clarkson (Rescheduled from 8/8/2025)" becomes "Kelly Clarkson").
2. Looks each artist up on **Last.fm** (listeners, play count, genre tags, on tour) and **MusicBrainz** (year formed, solo or group, country; **optional, off by default**, see `USE_MUSICBRAINZ`).
3. Saves one row per artist to `data/artists.csv`.

**Where it fits:** `01_music_db.py` builds the database, then the shows table is built (to-do step 1), then **this notebook** (step 2), then the venue lookup (step 3). Finally, the shows, artists and venues tables are joined on `artist_key` for the baseline model (step 7). The artist columns, especially `log_listeners` (popularity), `top_tag` (genre) and `years_active`, become model features.

---

**Input:** `data/music.db` (table `gametime`, built by `01_music_db.py`). Show names are cleaned into artist names here.
**Output:** `data/artists.csv`, one row per `artist_key` (the join key to the shows table)

| Source | What we get | Speed |
|---|---|---|
| Last.fm `artist.getInfo` | listeners, playcount, genre tags, on tour, MusicBrainz ID | ~4 calls/sec |
| MusicBrainz (optional, off by default) | year formed, solo vs. group, country | **1 call/sec** (their rule) |

**Reuses old lookups.** Answers are saved in `data/ticketmaster/interim/` (`lastfm.jsonl`, `musicbrainz.jsonl`). Any Gametime artist already looked up in the Ticketmaster era is skipped, so only new names take time.

**Before running**, `.env` needs:
```
LASTFM_API_KEY=your_key
MB_CONTACT=your_email@virginia.edu   # MusicBrainz asks apps to include a contact
```

**Runtime:** only new artists are fetched (about 15 min per 3,000 for Last.fm, about 1 hour per 3,000 for MusicBrainz, only if `USE_MUSICBRAINZ = True`). You can stop any time and re-run to pick up where it left off.
Set `LIMIT = 50` in the next cell to try it on a few artists first.

> **Note:** the Gametime shows table (to-do step 1) is not built yet. Until it is, section 1 builds the artist list straight from `music.db` using the cleaning rules from the to-do list (observed days with prices, no festival / multi-day passes, strip "Rescheduled from...", "18+"). Gametime has no state column, so US-only filtering is not applied here; non-US artists will just get a lookup too.

In [1]:
import json
import os
import re
import sqlite3
import time
import unicodedata
from difflib import SequenceMatcher
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from dotenv import load_dotenv

load_dotenv(".env", override=True)
LASTFM_KEY = os.getenv("LASTFM_API_KEY")
MB_CONTACT = os.getenv("MB_CONTACT", "student project")
assert LASTFM_KEY, "Add LASTFM_API_KEY to .env"

USE_MUSICBRAINZ = False  # False = skip the slow MusicBrainz lookup (year formed, solo/group, country stay blank
                         # except for artists already in the cache). True = fetch it (~1 hour per 3,000 new artists)
LIMIT = None             # e.g. 50 to test on a few artists first
SKIP_THEME_NIGHTS = True # tribute/party nights have no single artist to look up
CURRENT_YEAR = 2026

DB_PATH = Path("data/music.db")
OUT_PATH = Path("data/artists.csv")
INTERIM = Path("data/ticketmaster/interim")   # shared cache of saved API answers
INTERIM.mkdir(parents=True, exist_ok=True)
LFM_CACHE = INTERIM / "lastfm.jsonl"
MB_CACHE = INTERIM / "musicbrainz.jsonl"
print("Last.fm key found:", bool(LASTFM_KEY), "| MusicBrainz contact:", MB_CONTACT)

Last.fm key found: True | MusicBrainz contact: student project


## 1. List the artists to look up
Reads one row per Gametime show from `music.db`, then cleans the show name into an artist name.

In [2]:
# One row per show (eventId) that has at least one observed day with a price
with sqlite3.connect(DB_PATH) as conn:
    shows = pd.read_sql(
        """SELECT eventId AS event_id, eventName AS name, venueName AS venue, venueCity AS city
           FROM gametime
           WHERE dayState = 'observed' AND minPrice IS NOT NULL
           GROUP BY eventId""", conn)
print(f"{len(shows):,} shows with prices")

# Festival passes / multi-day tickets are not one artist playing one night
PASS = re.compile(r"\b(?:\d+[- ]day|day pass|weekend pass|season pass|festival|fest|"
                  r"(?:single|multi)[- ]day|(?:general admission|ga|vip) pass|pass)\b", re.I)
# Tribute / party nights have no single artist (kept narrow so "Night Ranger" survives)
THEME = re.compile(r"tribute|dance party|\bparty\b|brunch|happy hour|karaoke|open mic|jam session|"
                   r"showcase|celebration of|emo nite|drag |bingo|comedy", re.I)
# Show info that is not part of the artist name
NOISE = re.compile(r"\((?:rescheduled|moved|postponed|new date|new venue|late show|early show|"
                   r"18\+|21\+|all ages|sold out)[^)]*\)|\*[^*]*\*|\[[^\]]*\]|"
                   r"\b(?:rescheduled|moved|postponed) from .*$|\b(?:18|21)\+(?: event| show)?", re.I)

SEP = re.compile(
    r"\s+(?:w/|with|feat\.?|featuring|ft\.?|x|vs\.?|\+|\|)\s+"   # 'X w/ Y'
    r"|,|;"                                                         # lists
    r"|\s/\s?|\s?/\s"                                               # ' / ' (keeps AC/DC)
    r"|\s+[-–—]\s+"                                       # ' - Tour Name'
    r"|:\s", re.I)
TRAILING = re.compile(r"\s+(?:album release|record release|ep release|release show|residency.*|live|"
                      r"the .*tour|.*\btour\b.*|\d{4})$", re.I)
ENSEMBLE = re.compile(r"\s+(trio|quartet|quintet|sextet|septet|octet|nonet|big band|orchestra|band|ensemble)$", re.I)


def parse_headliner(name):
    if not isinstance(name, str):
        return None
    s = NOISE.sub(" ", name)
    s = re.sub(r"\([^)]*\)", " ", s).replace("�", "'").strip()
    m = re.search(r"\b(presents?):?\s+(.+)", s, re.I)
    if m:
        before, word, after = s[:m.start()].strip(), m.group(1).lower(), m.group(2).strip()
        artist_first = word == "present" or after.startswith(('"', "'")) or "tour" in after.lower()
        s = before if (artist_first and before) else after
    first = SEP.split(s, maxsplit=1)[0]
    first = TRAILING.sub("", first).strip(" -:,.'\"")
    return re.sub(r"\s{2,}", " ", first) or None


def make_key(name):
    """Join key: no accents, lowercase, letters and digits only, no leading 'the'.
    Same recipe as the Ticketmaster-era events table, so old cached lookups still match."""
    if not isinstance(name, str) or not name.strip():
        return None
    s = unicodedata.normalize("NFKD", name).encode("ascii", "ignore").decode()
    s = re.sub(r"[^a-z0-9]+", " ", s.lower().replace("&", " and ")).strip()
    return re.sub(r"^the\s+", "", s) or None


shows["is_pass"] = shows["name"].str.contains(PASS, na=False)
shows["is_theme_night"] = shows["name"].str.contains(THEME, na=False)
shows["artist_name"] = shows["name"].map(parse_headliner)
shows["alt_name"] = shows["artist_name"].str.replace(ENSEMBLE, "", regex=True).str.strip()   # 'Dan Tepfer Trio' -> 'Dan Tepfer'
shows["artist_key"] = shows["artist_name"].map(make_key)
print(f"dropped: {shows['is_pass'].sum():,} festival/pass shows, "
      f"{(shows['is_theme_night'] & ~shows['is_pass']).sum():,} theme nights, "
      f"{(shows['artist_key'].isna() & ~shows['is_pass']).sum():,} with no parsable artist")

ev = shows[~shows["is_pass"] & shows["artist_key"].notna()]
if SKIP_THEME_NIGHTS:
    ev = ev[~ev["is_theme_night"]]


def most_common(s):
    return s.mode().iat[0] if not s.mode().empty else s.iat[0]


artists = (ev.groupby("artist_key")
             .agg(artist_name=("artist_name", most_common),
                  alt_name=("alt_name", most_common),
                  n_shows=("event_id", "size"),
                  n_venues=("venue", "nunique"))
             .reset_index()
             .sort_values("n_shows", ascending=False))
if LIMIT:
    artists = artists.head(LIMIT)
print(f"{len(artists):,} artists to look up")
artists.head(10)

46,180 shows with prices
dropped: 518 festival/pass shows, 1,482 theme nights, 0 with no parsable artist
5,139 artists to look up


,artist_key,artist_name,alt_name,n_shows,n_venues
993,cirque du soleil mad apple,Cirque du Soleil Mad Apple,Cirque du Soleil Mad Apple,146,1
4546,tape face,Tape Face,Tape Face,115,3
1929,grand ole opry,Grand Ole Opry,Grand Ole Opry,101,1
3238,midnight,Midnight,Midnight,96,29
4710,trans siberian orchestra,Trans-Siberian Orchestra,Trans-Siberian,90,50
4339,soul of motown,Soul Of Motown,Soul Of Motown,85,1
1121,criss angel,Criss Angel,Criss Angel,84,1
2899,lollapalooza aftershow,Lollapalooza Aftershow,Lollapalooza Aftershow,77,25
691,boston symphony orchestra,Boston Symphony Orchestra,Boston Symphony,76,2
2915,los angeles philharmonic,Los Angeles Philharmonic,Los Angeles Philharmonic,70,2


In [3]:
# How many of these artists were already looked up in the Ticketmaster era? (the shared cache is reused)
_cache_keys = set()
for _f in (LFM_CACHE, MB_CACHE):
    if _f.exists():
        with _f.open(encoding="utf-8") as _fh:
            _cache_keys |= {json.loads(l)["artist_key"] for l in _fh if l.strip()}
hit = artists["artist_key"].isin(_cache_keys)
print(f"already cached: {hit.sum():,} artists ({artists.loc[hit, 'n_shows'].sum() / artists['n_shows'].sum():.0%} of shows) | new to fetch: {(~hit).sum():,}")

already cached: 5,139 artists (100% of shows) | new to fetch: 0


## 2. Helpers: cache, polite requests, name matching

In [4]:
def load_cache(path):
    if not path.exists():
        return {}
    with path.open(encoding="utf-8") as f:
        rows = [json.loads(line) for line in f if line.strip()]
    return {r["artist_key"]: r for r in rows}


def append_cache(path, row):
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(row) + "\n")


def get_json(url, params=None, headers=None, pause=0.25, tries=4):
    """GET with retries on rate limits and server hiccups. Returns (status_code, json or None)."""
    for attempt in range(tries):
        try:
            r = requests.get(url, params=params, headers=headers, timeout=20)
        except requests.RequestException:
            time.sleep(2 * (attempt + 1))
            continue
        time.sleep(pause)
        if r.status_code in (429, 500, 502, 503):
            time.sleep(3 * (attempt + 1))
            continue
        try:
            return r.status_code, r.json()
        except ValueError:
            return r.status_code, None
    return None, None


def norm(s):
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode().lower()
    s = re.sub(r"[^a-z0-9]+", " ", s.replace("&", " and ")).strip()
    return re.sub(r"^the\s+", "", s)


def similarity(a, b):
    return SequenceMatcher(None, norm(a), norm(b)).ratio()

## 3. Last.fm

Tries the artist name first, then the name without *Trio / Quartet / Big Band* if Last.fm doesn't know it. `autocorrect=1` fixes small spelling differences; the `name_similarity` column lets you catch wrong matches.

In [5]:
LFM_URL = "https://ws.audioscrobbler.com/2.0/"


def lastfm_lookup(name):
    status, js = get_json(LFM_URL, {"method": "artist.getinfo", "artist": name, "api_key": LASTFM_KEY,
                                    "autocorrect": 1, "format": "json"})
    if js is None:
        return "error", None
    if "error" in js:
        if js["error"] == 6:          # artist not found
            return "not_found", None
        if js["error"] in (10, 26):   # bad or suspended key: stop everything
            raise RuntimeError(f"Last.fm key problem: {js.get('message')}")
        return "error", js
    return "ok", js.get("artist")


cache = load_cache(LFM_CACHE)
todo = artists[~artists["artist_key"].isin(cache)]
print(f"Last.fm: {len(cache)} cached, {len(todo)} to fetch")

t0 = time.time()
for i, row in enumerate(todo.itertuples(), 1):
    status, data, used = "not_found", None, row.artist_name
    for name in dict.fromkeys([row.artist_name, row.alt_name]):   # unique, in order
        if not isinstance(name, str) or not name.strip():
            continue
        status, data = lastfm_lookup(name)
        used = name
        if status == "ok":
            break
    append_cache(LFM_CACHE, {"artist_key": row.artist_key, "query": used, "status": status, "data": data})
    if i % 100 == 0:
        print(f"  {i}/{len(todo)} | {(time.time() - t0) / 60:.1f} min")
print("Last.fm done.")

Last.fm: 7718 cached, 0 to fetch
Last.fm done.


In [6]:
def parse_lastfm(r):
    a = r.get("data") or {}
    tags = (a.get("tags") or {}).get("tag") or []
    if isinstance(tags, dict):
        tags = [tags]
    tag_names = [t.get("name", "").lower() for t in tags]
    stats = a.get("stats") or {}
    return {
        "artist_key": r["artist_key"],
        "lastfm_status": r["status"],
        "lastfm_query": r["query"],
        "lastfm_name": a.get("name"),
        "mbid": a.get("mbid") or None,
        "listeners": pd.to_numeric(stats.get("listeners"), errors="coerce"),
        "playcount": pd.to_numeric(stats.get("playcount"), errors="coerce"),
        "on_tour": a.get("ontour") == "1",
        "tags": "; ".join(tag_names[:5]) or None,
        "top_tag": tag_names[0] if tag_names else None,
    }


lfm = pd.DataFrame(parse_lastfm(r) for r in load_cache(LFM_CACHE).values())
lfm["name_similarity"] = [similarity(q, n) if isinstance(n, str) else np.nan
                          for q, n in zip(lfm["lastfm_query"], lfm["lastfm_name"])]
print(lfm["lastfm_status"].value_counts().to_string())
print("Has an mbid:", lfm["mbid"].notna().mean().round(2))

lastfm_status
ok           6387
not_found    1331
Has an mbid: 0.63


## 4. MusicBrainz
**Optional.** Controlled by `USE_MUSICBRAINZ` in the setup cell (off by default). When off, nothing new is fetched; artists already in the cache still get their MusicBrainz columns, everyone else is left blank.

Looks up by `mbid` when Last.fm gave one; otherwise searches by name and only accepts a near-exact match (score 95+).

In [7]:
MB_BASE = "https://musicbrainz.org/ws/2/artist"
MB_HEADERS = {"User-Agent": f"DS6021TicketPriceProject/0.1 ( {MB_CONTACT} )", "Accept": "application/json"}


def mb_lookup(mbid, name):
    if mbid:
        status, js = get_json(f"{MB_BASE}/{mbid}", {"fmt": "json"}, MB_HEADERS, pause=1.1)
        if status == 200 and js:
            return "ok_mbid", js
    if isinstance(name, str) and name.strip():
        q = 'artist:"' + name.replace('"', "") + '"'
        status, js = get_json(MB_BASE, {"query": q, "fmt": "json", "limit": 3}, MB_HEADERS, pause=1.1)
        hits = (js or {}).get("artists") or []
        if hits and int(hits[0].get("score", 0)) >= 95:
            return "ok_search", hits[0]
        return "not_found", None
    return "not_found", None


lfm_by_key = lfm.set_index("artist_key")
mb_cache = load_cache(MB_CACHE)
todo = artists[~artists["artist_key"].isin(mb_cache)]
if not USE_MUSICBRAINZ:
    print("USE_MUSICBRAINZ = False: skipping new MusicBrainz lookups")
    todo = todo.iloc[0:0]
print(f"MusicBrainz: {len(mb_cache)} cached, {len(todo)} to fetch (~{len(todo) * 1.2 / 60:.0f} min)")

t0 = time.time()
for i, row in enumerate(todo.itertuples(), 1):
    info = lfm_by_key.loc[row.artist_key] if row.artist_key in lfm_by_key.index else None
    mbid = info["mbid"] if info is not None and isinstance(info["mbid"], str) else None
    name = info["lastfm_name"] if info is not None and isinstance(info["lastfm_name"], str) else row.artist_name
    status, data = mb_lookup(mbid, name)
    append_cache(MB_CACHE, {"artist_key": row.artist_key, "status": status, "data": data})
    if i % 100 == 0:
        print(f"  {i}/{len(todo)} | {(time.time() - t0) / 60:.1f} min")
print("MusicBrainz done." if USE_MUSICBRAINZ else "MusicBrainz skipped.")

USE_MUSICBRAINZ = False: skipping new MusicBrainz lookups
MusicBrainz: 3377 cached, 0 to fetch (~0 min)
MusicBrainz skipped.


In [8]:
def parse_mb(r):
    a = r.get("data") or {}
    begin = (a.get("life-span") or {}).get("begin") or ""
    year = int(begin[:4]) if begin[:4].isdigit() else np.nan
    return {"artist_key": r["artist_key"], "mb_status": r["status"], "mb_id": a.get("id"),
            "mb_type": a.get("type"), "country": a.get("country"), "year_formed": year}


MB_COLS = ["artist_key", "mb_status", "mb_id", "mb_type", "country", "year_formed"]
mb = pd.DataFrame([parse_mb(r) for r in load_cache(MB_CACHE).values()], columns=MB_COLS)
print(mb["mb_status"].value_counts().to_string() if len(mb) else "no MusicBrainz data")

mb_status
ok_mbid      1942
not_found     986
ok_search     449


## 5. Combine and save

In [9]:
out = (artists.merge(lfm, on="artist_key", how="left")
               .merge(mb, on="artist_key", how="left"))

out["mb_status"] = out["mb_status"].fillna("not_run")   # never looked up (MusicBrainz off or not cached)
out["log_listeners"] = np.log1p(out["listeners"])
out["plays_per_listener"] = out["playcount"] / out["listeners"].replace(0, np.nan)
out["years_active"] = (CURRENT_YEAR - out["year_formed"]).clip(lower=0)
out["artist_type"] = out["mb_type"].map({"Person": "solo", "Group": "group", "Orchestra": "group",
                                         "Choir": "group"}).fillna("unknown")
out["mbid"] = out["mbid"].fillna(out["mb_id"])
out["low_confidence_match"] = out["name_similarity"] < 0.6

cols = ["artist_key", "artist_name", "n_shows", "n_venues", "lastfm_status", "lastfm_name", "name_similarity",
        "low_confidence_match", "listeners", "log_listeners", "playcount", "plays_per_listener",
        "on_tour", "tags", "top_tag", "mbid", "mb_status", "artist_type", "country",
        "year_formed", "years_active"]
out = out[cols]
out.to_csv(OUT_PATH, index=False)

print(f"Saved {OUT_PATH}: {len(out)} artists")
print(f"  Last.fm found:      {out['lastfm_status'].eq('ok').mean():.0%}")
print(f"  MusicBrainz found:  {out['mb_status'].str.startswith('ok').mean():.0%}")
print(f"  has year_formed:    {out['year_formed'].notna().mean():.0%}")
print(f"  low-confidence:     {out['low_confidence_match'].sum()} (check below)")
shows_covered = ev["artist_key"].isin(out.loc[out["lastfm_status"].eq("ok"), "artist_key"]).mean()
print(f"  share of shows with Last.fm data: {shows_covered:.0%}")

Saved data\artists.csv: 5139 artists
  Last.fm found:      87%
  MusicBrainz found:  15%
  has year_formed:    11%
  low-confidence:     31 (check below)
  share of shows with Last.fm data: 94%


In [10]:
# Possible wrong matches: query vs. what Last.fm returned. Fix by editing or dropping these rows.
out.loc[out["low_confidence_match"], ["artist_name", "lastfm_name", "name_similarity", "listeners"]].head(25)

,artist_name,lastfm_name,name_similarity,listeners
29,Musiq Soulchild,Musiq,0.500000,402237.0
642,The Lord of the Rings,Howard Shore,0.413793,1381300.0
895,Avenged Sevenfold and Good Charlotte,Good Charlotte,0.560000,2765373.0
1055,Eivor,Eivør Pálsdóttir,0.500000,30328.0
1240,Lady A,Lady Antebellum,0.571429,1118303.0
1461,Pat Benatar and Neil Giraldo,Pat Benatar,0.564103,1496558.0
1541,Sound Tribe Sector 9,STS9,0.333333,155869.0
1711,Joe Hisaishi,久石譲,0.000000,173483.0
2062,Morris Day and the Time,The Time,0.296296,170530.0
2152,Acid Mothers Temple,Acid Mothers Temple & The Melting Paraiso U.F.O.,0.558824,68926.0


In [11]:
# Sanity check: most popular artists in the data
out.nlargest(15, "listeners")[["artist_name", "listeners", "top_tag", "artist_type", "years_active", "n_shows"]]

,artist_name,listeners,top_tag,artist_type,years_active,n_shows
1930,Queen,7423239.0,classic rock,unknown,NaN,8
519,Gorillaz,7076765.0,electronic,unknown,NaN,22
154,Muse,6596439.0,alternative rock,unknown,NaN,33
1016,Foo Fighters,6550094.0,rock,unknown,NaN,15
854,The Strokes,5894730.0,indie rock,unknown,NaN,17
172,Weezer,5684067.0,alternative rock,unknown,NaN,32
331,The Weeknd,5587289.0,rnb,unknown,NaN,26
2402,Jay-Z,5543197.0,hip-hop,unknown,NaN,5
68,Bruno Mars,5494536.0,pop,unknown,NaN,41
4253,MGMT,5437286.0,electronic,unknown,NaN,1
